# Ranking the effectiveness of drugs on glioblastoma cell lines

**Reference:** **Tibor Fekete, J., & Győrffy, B. (2022). A unified platform enabling biomarker ranking and validation for 1562 drugs using transcriptomic data of 1250 cancer cell lines.** _Computational and Structural Biotechnology Journal_, 20, **2885–2894**. https://doi.org/10.1016/j.csbj.2022.06.007
 @tiborfeketeUnifiedPlatformEnabling2022

This notebook implements the workflow described by Fekete and Győrffy (2022):
1. Filter expression data for gliomas.
2. Filter sensitivity data for gliomas.
3. Compare datasets.
4. Label the lower response tertile as **sensitive** and the upper tertile as **resistant**; omit the middle tertile. _(Or something smarter)_
5. Randomly split samples into 66% training and 34% testing sets.
6. Select candidate genes with a training-set Mann-Whitney U test at $p<0.05$.
7. Combine selected genes in a random-forest classifier.
8. Evaluate the untouched test set using ROC AUC, accuracy, Cohen's kappa, sensitivity, specificity, and precision.

The implementation also includes single-gene ROC/Mann-Whitney results, Spearman correlations, Benjamini-Hochberg FDR, cell-line ranking, and a correlation matrix.

In [1]:
import pandas as pd
import numpy as np
import qnorm
import csv
import re
from collections import defaultdict
from decimal import Decimal
from scipy.stats import mannwhitneyu
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, precision_score

## 1. Load and filter expression data

- Model.cvs was used for identification of cell line oncotree lineages
- Source: https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2026Q1&filename=Model.csv
- Two expression datasets were used:

### 1.1 CCLE/DepMap/CTRP RNA-seq
- **Raw Data:** `Expression_(Short-read)_Public_26Q1_subsetted.csv`
- **Source:** https://depmap.org/portal/download/all/?release=DepMap+Public+26Q1&file=OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv&tab=allData

In [2]:
# Data import
ccle_exp = pd.read_csv(
    "../raw_data/expression/glioma/Expression_(Short-read)_Public_26Q1_subsetted.csv",
    index_col=0
)

In [3]:
ccle_exp.iloc[:5, :5]

,RBM5,RBM6,SLC25A13,CACNG3,ZNF263
ACH-001113,5.591937,5.578833,3.518019,0.000000,4.832774
ACH-001289,6.323073,5.533597,4.864454,0.024853,5.095810
ACH-001339,5.248755,4.470589,5.362344,0.000000,3.891711
ACH-001979,5.015364,3.445246,4.436162,0.000000,5.127530
ACH-002438,5.085331,4.914588,3.542815,0.000000,4.618145


In [4]:
model = pd.read_csv(
    "../raw_data/annotation/Model.csv",
    index_col=0
)

In [5]:
model.iloc[:200, :200]

,PatientID,CellLineName,StrippedCellLineName,DepmapModelType,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,OncotreeCode,PatientSubtypeFeatures,RRID,...,PublicComments,CCLEName,HCMIID,PediatricModelType,ModelAvailableInDbgap,ModelSubtypeFeatures,WTSIMasterCellID,SangerModelID,COSMICID,ModelIDAlias
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-000001,PT-gj46wT,NIH:OVCAR-3,NIHOVCAR3,HGSOC,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,HGSOC,NaN,CVCL_0465,...,NaN,NIHOVCAR3_OVARY,NaN,False,Approved for public sharing - CCLE,NaN,2201.0,SIDM00105,905933.0,NaN
ACH-000002,PT-5qa3uk,HL-60,HL60,AMLMRC,Myeloid,Acute Myeloid Leukemia,AML with Myelodysplasia-Related Changes,AMLMRC,"TP53(del), CDKN2A and NRAS mutations [PubMed=2...",CVCL_0002,...,NaN,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,"NRAS, BCOR and CDKN2A",55.0,SIDM00829,905938.0,NaN
ACH-000003,PT-puKIyc,CACO2,CACO2,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,CVCL_0025,...,NaN,CACO2_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00891,NaN,NaN
ACH-000004,PT-q4K2cp,HEL,HEL,AMLNOS,Myeloid,Acute Myeloid Leukemia,"AML, NOS",AMLNOS,JAK2 and TP53 mutations,CVCL_0001,...,NaN,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,JAK2 and TP53,783.0,SIDM00594,907053.0,NaN
ACH-000005,PT-q4K2cp,HEL 92.1.7,HEL9217,AML,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,AML,JAK2 and TP53 mutations,CVCL_2481,...,NaN,HEL9217_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,NaN,NaN,SIDM00593,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000197,PT-XChmHC,TALL-1,TALL1,TALL,Lymphoid,T-Lymphoblastic Leukemia/Lymphoma,T-Cell Acute Lymphoblastic Leukemia,NaN,"STAG2-PTGER3,UHRF1-PLIN3, NRAS and TP53 mutations",CVCL_1736,...,NaN,TALL1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,"TAL1, TP53",2218.0,SIDM00370,909762.0,NaN
ACH-000198,PT-FYT6ze,EOL-1,EOL1,AML,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,AML,"FIP1L1::PDGFRA, KMT2A partial tandem duplication",CVCL_0258,...,NaN,EOL1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,"FIP1L1::PDGFRA, KMT2A-PTD, PHF6",426.0,SIDM00277,906856.0,NaN
ACH-000199,PT-B6GmiY,Hs 255.T,HS255T,ZFIBCOL,Fibroblast,Non-Cancerous,"Fibroblast, Colorectal",NaN,NaN,CVCL_0707,...,NaN,HS255T_FIBROBLAST,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM01613,NaN,NaN


In [6]:
# Add metadata to the expression data
ccle_exp.index.name = "ModelID"

metadata_columns = [
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype",
]

for position, column in enumerate(metadata_columns):
    ccle_exp.insert(
        position,
        column,
        model[column].reindex(ccle_exp.index),
    )

In [7]:
print("Unmatched models:", ccle_exp["OncotreeLineage"].isna().sum())

Unmatched models: 12


In [8]:
# For 12 rows the OncotreeLineage was labeled NA
# Remove these rows
ccle_exp = ccle_exp.dropna(subset=["OncotreeLineage"])

In [9]:
ccle_exp.iloc[:200, :200]

,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,RBM5,RBM6,SLC25A13,CACNG3,ZNF263,CCL26,USH1C,...,DNPH1,PRSS16,KIF20A,CDH9,APBB3,CERT1,POLR3G,CDH6,SLC12A7,RAD50
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-001113,Lung,Non-Small Cell Lung Cancer,Lung Squamous Cell Carcinoma,5.591937,5.578833,3.518019,0.000000,4.832774,1.269291,0.276878,...,4.749384,3.794333,4.750681,0.000000,4.881063,4.414744,2.455860,0.925937,3.583952,4.232646
ACH-001289,CNS/Brain,Embryonal Tumor,Atypical Teratoid/Rhabdoid Tumor,6.323073,5.533597,4.864454,0.024853,5.095810,0.000000,0.073646,...,6.211497,3.481596,5.501506,0.034178,5.011482,5.107679,1.999762,0.038210,2.986154,5.269370
ACH-001339,Skin,Melanoma,Cutaneous Melanoma,5.248755,4.470589,5.362344,0.000000,3.891711,0.252796,0.012451,...,4.407971,0.303334,5.859637,0.019885,2.621977,5.772039,5.113521,0.080579,3.219269,3.643868
ACH-001979,Skin,Melanoma,Melanoma,5.015364,3.445246,4.436162,0.000000,5.127530,0.000000,0.000000,...,4.544739,2.710814,2.300305,0.000000,1.666729,5.370063,4.448095,0.000000,4.154761,4.860670
ACH-002438,Kidney,Wilms' Tumor,Wilms' Tumor,5.085331,4.914588,3.542815,0.000000,4.618145,1.969127,1.329652,...,4.330501,0.000000,5.917816,0.000000,2.236309,3.914182,1.939087,0.184972,2.799699,4.148999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000858,Lung,Non-Small Cell Lung Cancer,Lung Squamous Cell Carcinoma,4.625791,3.986620,4.781417,0.000000,4.269961,0.750315,0.000000,...,5.223295,2.493376,3.940308,0.000000,2.138097,3.462537,2.920495,6.089793,2.989889,3.362209
ACH-001421,Eye,Retinoblastoma,Retinoblastoma,6.484462,6.255697,5.637315,2.975227,5.194717,1.601354,0.000000,...,6.744941,0.301144,5.463171,0.000000,4.747216,3.749019,2.691695,0.000000,4.616210,4.180554
ACH-002796,Soft Tissue,CIC-DUX4 Sarcoma,CIC-DUX4 Sarcoma,6.103065,5.809877,4.242321,0.000000,4.816528,0.078420,0.000000,...,4.877906,0.618601,4.460205,0.631748,3.705867,4.902754,1.736730,1.875202,3.224340,1.240388


In [10]:
ccle_brain_exp = ccle_exp[
    ccle_exp["OncotreeLineage"].eq("CNS/Brain")
].copy()

In [11]:
print(ccle_brain_exp.shape)
print("CCLE cell lines:", ccle_brain_exp["OncotreeLineage"].count())

(103, 19218)
CCLE cell lines: 103


### 1.2 **GDSC1/GDSC2 RNA-normalized Affymetrix HGU-219 Expression Matrix:**
- As of 19/7/2026, the preprocessed expression data released by the Sanger Institute is not available on the original website anymore.
- This source was used to trace back the original link: https://github.com/BorgwardtLab/Kernelized-Rank-Learning/blob/master/URLs_to_GDSC_datasets.txt
- **Raw Data ArrayExpress Accession:** `E-MTAB-3610`
- **Preprocessed Data:** `Cell_line_RMA_proc_basalExp.txt`
- **Original Source:** http://www.cancerrxgene.org/gdsc1000/GDSC1000_WebResources/Data/preprocessed/Cell_line_RMA_proc_basalExp.txt.zip
- **Currently Available Source:** https://web.archive.org/web/20260209143748/http://www.cancerrxgene.org/gdsc1000/GDSC1000_WebResources/Data/preprocessed/Cell_line_RMA_proc_basalExp.txt.zip

In [1]:
# Data import
gdsc_exp = pd.read_csv(
    "../raw_data/expression/Cell_line_RMA_proc_basalExp.txt",
    sep="\t",
    low_memory=False
)

NameError: name 'pd' is not defined

In [13]:
gdsc_exp.iloc[:5, :5]

,GENE_SYMBOLS,GENE_title,DATA.906826,DATA.687983,DATA.910927
0,TSPAN6,tetraspanin 6 [Source:HGNC Symbol;Acc:11858],7.632023,7.548671,8.712338
1,TNMD,tenomodulin [Source:HGNC Symbol;Acc:17757],2.964585,2.777716,2.643508
2,DPM1,dolichyl-phosphate mannosyltransferase polypep...,10.379553,11.807341,9.880733
3,SCYL3,SCY1-like 3 (S. cerevisiae) [Source:HGNC Symbo...,3.614794,4.066887,3.956230
4,C1orf112,chromosome 1 open reading frame 112 [Source:HG...,3.380681,3.732485,3.236620


In [14]:
gdsc_exp = (
    gdsc_exp
    .dropna(subset=["GENE_SYMBOLS"])
    .drop(columns=["GENE_title"])
    .set_index("GENE_SYMBOLS")
    .T
)

gdsc_exp.index = (
    gdsc_exp.index.astype(str)
    .str.replace(r"^DATA\.", "", regex=True)
)

gdsc_exp.columns.name = None
gdsc_exp.index.name = "COSMICID"

In [15]:
metadata_columns = [
    "ModelID",
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype",
]

gdsc_exp.index = gdsc_exp.index.astype(str)

model_cosmic = model.dropna(subset=["COSMICID"]).copy()
model_cosmic = model_cosmic.reset_index()
model_cosmic["COSMICID"] = (
    model_cosmic["COSMICID"]
    .astype("Int64")
    .astype(str)
)

metadata = (
    model_cosmic
    .drop_duplicates(subset="COSMICID")
    .set_index("COSMICID")[metadata_columns]
    .reindex(gdsc_exp.index)
)

gdsc_exp = gdsc_exp.drop(
    columns=[col for col in metadata_columns if col in gdsc_exp.columns]
)

gdsc_exp = metadata.join(gdsc_exp)

gdsc_exp.index.name = "COSMICID"

In [16]:
gdsc_exp.iloc[:200, :5]

,ModelID,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,TSPAN6
COSMICID,,,,,
906826,ACH-000212,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,7.632023
687983,ACH-000530,Lung,SMARCA4-deficient undifferentiated tumor,SMARCA4-deficient undifferentiated tumor,7.548671
910927,ACH-000856,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,8.712338
1240138,ACH-002135,Pleura,Pleural Mesothelioma,"Pleural Mesothelioma, Biphasic Type",7.797142
1240139,ACH-002136,Pleura,Pleural Mesothelioma,Pleural Mesothelioma,7.729268
...,...,...,...,...,...
907300,ACH-002262,Myeloid,Acute Myeloid Leukemia,Acute Myelomonocytic Leukemia,3.334582
907276,ACH-000793,Esophagus/Stomach,Esophagogastric Adenocarcinoma,Signet Ring Cell Carcinoma of the Stomach,7.448062
907277,ACH-000101,Lymphoid,T-Lymphoblastic Leukemia/Lymphoma,T-Cell Acute Lymphoblastic Leukemia,6.160155


In [17]:
print("Unmatched models:", gdsc_exp["OncotreeLineage"].isna().sum())

Unmatched models: 73


In [18]:
# For 73 rows the OncotreeLineage was labeled NA
# Remove these rows
gdsc_exp = gdsc_exp.dropna(subset=["OncotreeLineage"])

In [19]:
gdsc_brain_exp = gdsc_exp[
    gdsc_exp["OncotreeLineage"].eq("CNS/Brain")
].copy()

In [20]:
print(gdsc_brain_exp.shape)
print("GDSC cell lines:", gdsc_brain_exp["OncotreeLineage"].count())

(56, 17423)
GDSC cell lines: 56


## 1.3 Filtering
Reference: **Zhang, C., Neha, Zhang, J., Dhaha, P., Li, X., Mishra, S. K., Fleishman, J., Parvez, S., Jha, S. K., & Huang, M. (2025). Aging and senescence: Key players in brain tumor progression and drug resistance. Drug Resistance Updates, 81, 101228.** https://doi.org/10.1016/j.drup.2025.101228
 zhangAgingSenescenceKey2025

Primary types of gliomas according to Zhang et al.:
1. Astrocytomas: e.g.: Glioblastoma
2. Oligodendrogliomas
3. Ependymomas
4. Mixed gliomas

Cell lines with Brain/CNS OcotreeLineage:
- CCLE: 103
- GDSC: 56

In [21]:
print("CCLE cell lines:", ccle_brain_exp["OncotreeLineage"].count())
print("GDSC cell lines:", gdsc_brain_exp["OncotreeLineage"].count())
print("All cell lines:", (ccle_brain_exp["OncotreeLineage"].count()) + (gdsc_brain_exp["OncotreeLineage"].count()))

print("\nCCLE subtype counts:")
print(ccle_brain_exp["OncotreeSubtype"].value_counts(dropna=False))

print("\nGDSC subtype counts:")
print(gdsc_brain_exp["OncotreeSubtype"].value_counts(dropna=False))

CCLE cell lines: 103
GDSC cell lines: 56
All cell lines: 159

CCLE subtype counts:
OncotreeSubtype
Glioblastoma, IDH-Wildtype                     55
Astrocytoma, IDH-Mutant                        13
Medulloblastoma, NOS                            7
Atypical Teratoid/Rhabdoid Tumor                6
Meningioma                                      4
Primitive Neuroectodermal Tumor                 3
Medulloblastoma, Group 3                        3
Immortalized Astrocytes                         3
Gliosarcoma                                     2
Adult-Type Diffuse Glioma                       2
Immortalized Epithelial Cells, CNS              1
Immortalized Endothelial Cells, Endoneurial     1
Immortalized Neuronal Progenitor Cells          1
Astrocytoma, IDH-Mutant, Grade 3                1
Glioma, NOS                                     1
Name: count, dtype: int64

GDSC subtype counts:
OncotreeSubtype
Glioblastoma, IDH-Wildtype          34
Astrocytoma, IDH-Mutant             11
Gliosarco

In [22]:
matching_ids = sorted(
    set(gdsc_brain_exp["ModelID"].dropna().astype(str).str.strip()) &
    set(ccle_brain_exp.index.dropna().astype(str).str.strip())
)

print(f"Number of duplicate cell lines: {len(matching_ids)}")

Number of duplicate cell lines: 40


- There are 40 cell lines both present in the CCLE and GDSC databases.

In [23]:
print("CCLE cell lines:", ccle_brain_exp["OncotreeLineage"].count())
print("GDSC cell lines:", gdsc_brain_exp["OncotreeLineage"].count())
print("All unique cell lines:", len(ccle_brain_exp.index.union(gdsc_brain_exp['ModelID'])))

print("\nCCLE subtype counts:")
print(ccle_brain_exp["OncotreeSubtype"].value_counts(dropna=False))

print("\nGDSC subtype counts:")
print(gdsc_brain_exp["OncotreeSubtype"].value_counts(dropna=False))

CCLE cell lines: 103
GDSC cell lines: 56
All unique cell lines: 119

CCLE subtype counts:
OncotreeSubtype
Glioblastoma, IDH-Wildtype                     55
Astrocytoma, IDH-Mutant                        13
Medulloblastoma, NOS                            7
Atypical Teratoid/Rhabdoid Tumor                6
Meningioma                                      4
Primitive Neuroectodermal Tumor                 3
Medulloblastoma, Group 3                        3
Immortalized Astrocytes                         3
Gliosarcoma                                     2
Adult-Type Diffuse Glioma                       2
Immortalized Epithelial Cells, CNS              1
Immortalized Endothelial Cells, Endoneurial     1
Immortalized Neuronal Progenitor Cells          1
Astrocytoma, IDH-Mutant, Grade 3                1
Glioma, NOS                                     1
Name: count, dtype: int64

GDSC subtype counts:
OncotreeSubtype
Glioblastoma, IDH-Wildtype          34
Astrocytoma, IDH-Mutant             11
Gl

- CCLE cell lines: 63
- GDSC cell lines: 56
- All cell lines: 119

## 2. Load and filter sensitivity data
Checking if there's sensitivity data for all the expression data

### 2.1 CCLE
- AUC is normalised to 1
- **CTD^2 Raw Data:** `Drug_sensitivity_AUC_(CTD^2)_subsetted.csv`
- **DepMap Raw Data:** `Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv`
- **Sources:**
    - https://depmap.org/portal/download/all/?release=Harmonized+CTD%5E2+25Q2&file=CTRPAUCMatrix.csv&tab=allData
    - https://depmap.org/portal/download/all/?release=Harmonized+PRISM+Repurposing+Secondary+Screen+25Q2&file=REPURPOSINGAUCMatrix.csv&tab=allData

### 2.1.1 CTRPv2
- Supposedly CTD^2 is a new, unified database containing CTRPv2 data
- This claim was crosschecked with the old CTRPv2 data
- AUC isn't normalised
- **CTRPv2 Raw Data:**
    -  `CTRPv2.2_2015_pub_CancerDisc_5_1210.zip`
        - `v22.data.auc_sensitivities.txt`
        - `v22.meta.per_cell_line.txt`
        - `v22.meta.per_compound.txt`
- **Source:** https://d1hp0ufb0xxisr.cloudfront.net/CTD2/Broad/CTRPv2.2_2015_pub_CancerDisc_5_1210.zip

In [24]:
def read_tsv(filename):
    with open(filename, "r", newline="", encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle, delimiter="\t"))


def read_csv(filename):
    with open(filename, "r", newline="", encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle))

In [25]:
auc = pd.read_csv(
    "../raw_data/sensitivity/glioma/CTRPv2.2_2015_pub_CancerDisc_5_1210/v22.data.auc_sensitivities.txt",
    sep="\t",
)

cell_metadata = pd.read_csv(
    "../raw_data/sensitivity/glioma/CTRPv2.2_2015_pub_CancerDisc_5_1210/v22.meta.per_cell_line.txt",
    sep="\t",
)

compound_metadata = pd.read_csv(
    "../raw_data/sensitivity/glioma/CTRPv2.2_2015_pub_CancerDisc_5_1210/v22.meta.per_compound.txt",
    sep="\t",
)

model_metadata = pd.read_csv(
    "../raw_data/annotation/Model.csv",
    low_memory=False,
)

print("AUC records:", len(auc))
print("CTRP profiles:", len(cell_metadata))
print("Compounds:", len(compound_metadata))
print("PRISM models:", len(model_metadata))

AUC records: 260496
CTRP profiles: 664
Compounds: 481
PRISM models: 2154


In [26]:
def normalize_name(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper(),
    )

In [27]:
model_name_mapping = pd.concat(
    [
        model_metadata[["ModelID", "CellLineName"]].rename(
            columns={"CellLineName": "model_name"}
        ),
        model_metadata[["ModelID", "StrippedCellLineName"]].rename(
            columns={"StrippedCellLineName": "model_name"}
        ),
        model_metadata[["ModelID", "CCLEName"]]
        .assign(
            CCLEName=lambda x: x["CCLEName"]
            .fillna("")
            .str.split("_")
            .str[0]
        )
        .rename(columns={"CCLEName": "model_name"}),
    ],
    ignore_index=True,
)

model_name_mapping["normalized_name"] = (
    model_name_mapping["model_name"].map(normalize_name)
)

model_name_mapping = (
    model_name_mapping
    .loc[model_name_mapping["normalized_name"] != ""]
    .drop_duplicates(
        subset=["normalized_name", "ModelID"]
    )
)

model_name_mapping.head()

,ModelID,model_name,normalized_name
0,ACH-000001,NIH:OVCAR-3,NIHOVCAR3
1,ACH-000002,HL-60,HL60
2,ACH-000003,CACO2,CACO2
3,ACH-000004,HEL,HEL
4,ACH-000005,HEL 92.1.7,HEL9217


In [28]:
manual_overrides = {
    "COLO699": "ACH-001041",
    "HCC1588": "ACH-001078",
    "NCIH1339": "ACH-000921",
    "H292": "ACH-001075",
    "TTTHYROID": "ACH-001321",
}


def choose_model_id(ccl_name):
    normalized_name = normalize_name(ccl_name)

    if normalized_name in manual_overrides:
        return manual_overrides[normalized_name]

    matches = (
        model_name_mapping.loc[
            model_name_mapping["normalized_name"]
            == normalized_name,
            "ModelID",
        ]
        .dropna()
        .unique()
        .tolist()
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        original_matches = [
            model_id
            for model_id in matches
            if int(model_id.split("-")[1]) < 2000
        ]

        if len(original_matches) == 1:
            return original_matches[0]

    raise ValueError(
        f"Could not map {ccl_name!r}; "
        f"candidate ModelIDs: {matches}"
    )

In [29]:
cell_metadata["ModelID"] = (
    cell_metadata["ccl_name"].map(choose_model_id)
)

print(
    "CTRP profiles:",
    cell_metadata["index_ccl"].nunique(),
)

print(
    "Unique DepMap models:",
    cell_metadata["ModelID"].nunique(),
)

cell_metadata[
    ["index_ccl", "ccl_name", "ModelID"]
].head()

CTRP profiles: 664
Unique DepMap models: 645


,index_ccl,ccl_name,ModelID
0,1,CHP212,ACH-000120
1,2,IMR32,ACH-000310
2,3,KELLY,ACH-000259
3,4,KPNSI9S,ACH-000446
4,5,KPNYN,ACH-000227


In [30]:
compound_metadata["compound_label"] = (
    compound_metadata["cpd_name"].str.upper()
    + " (CTRP:"
    + compound_metadata["master_cpd_id"]
        .astype("Int64")
        .astype("string")
    + ")"
)
if compound_metadata["compound_label"].duplicated().any():
    raise ValueError("Compound labels are not unique.")

compound_metadata[
    ["index_cpd", "cpd_name", "compound_label"]
].head()

,index_cpd,cpd_name,compound_label
0,1,ML311,ML311 (CTRP:290356)
1,2,zebularine,ZEBULARINE (CTRP:411739)
2,3,BRD-A02303741,BRD-A02303741 (CTRP:659371)
3,4,importazole,IMPORTAZOLE (CTRP:46792)
4,5,SR8278,SR8278 (CTRP:668858)


In [31]:
annotated_auc = (
    auc
    .merge(
        cell_metadata[["index_ccl", "ModelID"]],
        on="index_ccl",
        how="left",
        validate="many_to_one",
    )
    .merge(
        compound_metadata[
            ["index_cpd", "compound_label"]
        ],
        on="index_cpd",
        how="left",
        validate="many_to_one",
    )
)

if annotated_auc["ModelID"].isna().any():
    raise ValueError("Some CTRP cell lines lack a ModelID.")

if annotated_auc["compound_label"].isna().any():
    raise ValueError("Some compounds lack annotations.")

annotated_auc.head()

,index_cpd,index_ccl,area_under_curve,ModelID,compound_label
0,1,1,11.025,ACH-000120,ML311 (CTRP:290356)
1,1,2,9.356,ACH-000310,ML311 (CTRP:290356)
2,1,3,10.734,ACH-000259,ML311 (CTRP:290356)
3,1,4,11.337,ACH-000446,ML311 (CTRP:290356)
4,1,5,14.064,ACH-000227,ML311 (CTRP:290356)


In [32]:
ctrpv2 = annotated_auc.pivot_table(
    index="ModelID",
    columns="compound_label",
    values="area_under_curve",
    aggfunc="mean",
)

ctrpv2 = ctrpv2.sort_index().sort_index(axis=1)

ctrpv2.index.name = None
ctrpv2.columns.name = None

ctrpv2.shape

(645, 481)

### 2.1.1 Crosschecking CCLE datasets

In [33]:
ctd2 = pd.read_csv(
    "../raw_data/sensitivity/glioma/Drug_sensitivity_AUC_(CTD^2)_subsetted.csv",
    index_col=0
)

prism = pd.read_csv(
    "../raw_data/sensitivity/glioma/Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv",
    index_col=0
)

ctrpv2.to_csv(
    "../raw_data/sensitivity/glioma/Drug_sensitivity_AUC_(CTRPv2.2)_wide.csv",
    index=True,
)
print("CTRP:", ctd2.shape, "\nPRISM:", prism.shape, "\nCTRPv2:", ctrpv2.shape)

CTRP: (810, 545) 
PRISM: (727, 1482) 
CTRPv2: (645, 481)


In [34]:
ctd2.iloc[:5, :5]

,(+)-JQ-1 (CTRP:616354),AZD4547 (CTRP:660325),16-BETA-BROMOANDROSTERONE (CTRP:411726),TANESPIMYCIN (CTRP:50134),"1S,3R-RSL-3 (CTRP:609060)"
ACH-000002,0.550735,NaN,0.993196,0.739044,0.693123
ACH-000004,0.489051,0.858347,0.928700,0.854403,0.291483
ACH-000005,0.518107,NaN,1.000000,0.746597,0.314420
ACH-000006,0.703030,0.866474,0.798646,0.776359,0.542607
ACH-000007,0.743717,0.877469,NaN,0.396336,0.603077


In [35]:
prism.iloc[:5, :200]

,(+)-CAMPTOTHECIN (BRD:BRD-K37890730-001-15-1),1-AZAKENPAULLONE (BRD:BRD-K03273112-001-01-8),1-NAPHTHYL-PP1 (BRD:BRD-K29542628-001-03-9),1-PHENYLBIGUANIDE (BRD:BRD-K31491153-003-05-9),10-DEACETYLBACCATIN (BRD:BRD-K96631475-001-02-4),10-HYDROXYCAMPTOTHECIN (BRD:BRD-K63784565-001-06-2),AZD4547 (BRD:BRD-K28392481-001-05-1),12-O-TETRADECANOYLPHORBOL-13-ACETATE (BRD:BRD-K68552125-001-05-3),TANESPIMYCIN (BRD:BRD-K81473043-001-19-5),2'-MECCPA (BRD:BRD-K63932022-001-01-5),...,BATIMASTAT (BRD:BRD-K82818427-001-04-8),BAY-11-7082 (BRD:BRD-K15025317-001-15-1),BAY-11-7085 (BRD:BRD-K48684885-001-04-2),BAY-87-2243 (BRD:BRD-K70463136-001-01-5),BAY-K-8644-(S)-(-) (BRD:BRD-K14329163-001-03-9),BECLOMETHASONE-DIPROPIONATE (BRD:BRD-K97810537-001-13-6),BEKANAMYCIN (BRD:BRD-K73425385-065-04-8),BELINOSTAT (BRD:BRD-K17743125-001-08-4),BENDAMUSTINE (BRD:BRD-K17068645-003-04-2),BENDROFLUMETHIAZIDE (BRD:BRD-A80017228-001-25-7)
ACH-000001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.796344,NaN,...,0.740303,NaN,NaN,NaN,NaN,NaN,NaN,0.572391,NaN,NaN
ACH-000004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.855552,NaN,...,0.822426,NaN,NaN,NaN,NaN,NaN,NaN,0.606195,NaN,NaN
ACH-000005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.802643,NaN,...,0.869777,NaN,NaN,NaN,NaN,NaN,NaN,0.633317,NaN,NaN
ACH-000006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.880611,NaN,...,0.936405,NaN,NaN,NaN,NaN,NaN,NaN,0.871476,NaN,NaN
ACH-000007,0.282169,0.898721,0.873003,0.856499,0.869975,0.259895,1.0,0.73199,0.865442,1.0,...,0.891419,0.89825,0.853291,0.581957,1.0,0.994094,0.993229,0.629858,0.984054,0.994473


In [36]:
ctrpv2.iloc[:5, :5]

,16-BETA-BROMOANDROSTERONE (CTRP:411726),"1S,3R-RSL-3 (CTRP:609060)",3-CL-AHPC (CTRP:660796),968 (CTRP:660080),A-804598 (CTRP:661033)
ACH-000004,14.079,4.5275,8.1324,14.871,NaN
ACH-000007,15.355,8.9164,9.6490,13.430,NaN
ACH-000008,14.803,9.2353,12.8000,NaN,15.367
ACH-000009,13.803,8.3567,13.2710,NaN,14.624
ACH-000011,14.839,7.5951,11.0770,NaN,NaN


In [37]:
for dataframe in [ctd2, ctrpv2, prism]:
    dataframe.index = (
        dataframe.index
        .astype(str)
        .str.strip()
        .str.upper()
    )

ccle_raw_counts = (ctd2.index
    .union(ctrpv2.index)
    .union(prism.index))

print("Unique cell lines across all CCLE datasets:", len(ccle_raw_counts))

Unique cell lines across all CCLE datasets: 933


In [38]:
print("CTD^2 rows:", len(ctd2))
print("CTRPv2 rows:", len(ctrpv2))
print("PRISM rows:", len(prism))

CTD^2 rows: 810
CTRPv2 rows: 645
PRISM rows: 727


In [39]:
ctd2 = ctd2.loc[ctd2.index.isin(ccle_brain_exp.index)].copy()
ctd2 = ctd2.dropna(axis=0, how="all")

ctrpv2 = ctrpv2.loc[ctrpv2.index.isin(ccle_brain_exp.index)].copy()
ctrpv2 = ctrpv2.dropna(axis=0, how="all")

prism = prism.loc[prism.index.isin(ccle_brain_exp.index)].copy()
prism = prism.dropna(axis=0, how="all")

In [40]:
assert ctd2.index.isin(ccle_brain_exp.index).all()
assert ctrpv2.index.isin(ccle_brain_exp.index).all()
assert prism.index.isin(ccle_brain_exp.index).all()

ccle_counts = (ctd2.index
    .union(ctrpv2.index)
    .union(prism.index))

print("CTD^2 glioma rows:", len(ctd2))
print("CTRPv2 glioma rows:", len(ctrpv2))
print("PRISM glioma rows", len(prism))
print("Unique cell lines across all CCLE datasets:", len(ccle_counts))


CTD^2 glioma rows: 47
CTRPv2 glioma rows: 47
PRISM glioma rows 55
Unique cell lines across all CCLE datasets: 61


### 2.2 GDSC
- AUC is normalised to 1
- **GDSC1 Raw Data:** `Drug_sensitivity_AUC_(Sanger_GDSC1)_subsetted.csv`
- **GDSC2 Raw Data:** `Drug_sensitivity_AUC_(Sanger_GDSC2)_subsetted.csv`
- **Sources:**
     - https://depmap.org/portal/download/all/?release=Harmonized+GDSC+25Q2&file=GDSC1AUCMatrix.csv&tab=allData
     - https://depmap.org/portal/download/all/?release=Harmonized+GDSC+25Q2&file=GDSC2AUCMatrix.csv&tab=allData

In [41]:
gdsc1 = pd.read_csv(
    "../raw_data/sensitivity/glioma/Drug_sensitivity_AUC_(Sanger_GDSC1)_subsetted.csv",
    index_col=0
)

gdsc2 = pd.read_csv(
    "../raw_data/sensitivity/glioma/Drug_sensitivity_AUC_(Sanger_GDSC2)_subsetted.csv",
    index_col=0
)

print("GDSC1:", gdsc1.shape, "\nGDSC2:", gdsc2.shape)

GDSC1: (948, 316) 
GDSC2: (947, 286)


In [42]:
gdsc1.iloc[:5, :5]

,(+)-JQ-1 (GDSC1:1218),(5Z)-7-OXOZEAENOL (GDSC1:1242),AZD4547 (GDSC1:1135),1080622-86-1 (GDSC1:152),212631-79-3 (GDSC1:1015)
ACH-000001,1.000000,0.786707,0.999868,0.891683,0.936068
ACH-000002,0.971357,0.749139,0.804602,0.775744,0.409988
ACH-000004,1.000000,0.737210,0.954284,0.646709,0.870338
ACH-000006,0.824947,0.204947,0.536506,0.447452,0.752790
ACH-000007,0.979148,0.566650,0.979903,0.890769,0.630178


In [43]:
gdsc2.iloc[:5, :5]

,(+)-CAMPTOTHECIN (GDSC2:1003),(+)-JQ-1 (GDSC2:2172),AZD4547 (GDSC2:1786),1044870-39-4 (GDSC2:1625),TANESPIMYCIN (GDSC2:1026)
ACH-000001,0.952757,0.831799,0.885178,0.966006,0.892516
ACH-000002,0.697881,0.421295,0.999851,0.969576,0.926131
ACH-000004,0.758526,0.572811,0.984711,0.998732,0.993055
ACH-000006,0.747942,0.472126,0.990952,NaN,0.762122
ACH-000007,0.778251,0.711866,0.964930,0.979922,0.683957


In [44]:
gdsc1 = gdsc1.loc[gdsc1.index.isin(gdsc_brain_exp["ModelID"])].copy()
gdsc2 = gdsc2.loc[gdsc2.index.isin(gdsc_brain_exp["ModelID"])].copy()

gdsc1 = gdsc1.dropna(axis=0, how="all")
gdsc2 = gdsc2.dropna(axis=0, how="all")

for dataframe in [gdsc1, gdsc2]:
    dataframe.index = (
        dataframe.index
        .astype(str)
        .str.strip()
        .str.upper()
    )
gdsc_counts = (gdsc1.index.union(gdsc2.index))

print("GDSC unique cell lines:", len(gdsc_counts))
print("GDSC1 glioma rows:", len(gdsc1.index))
print("GDSC2 glioma rows:", len(gdsc2.index))

GDSC unique cell lines: 54
GDSC1 glioma rows: 54
GDSC2 glioma rows: 54


## 3. Comparing CCLE and GDSC Datasets
- Evaluating the total number of unique cell lines
- 5 Datasets in total:
    - CCLE Datasets:
        - CTRPv2 (2015): AUC not normalised
        - CTD^2 (2026): AUC normalised to 1
        - PRISM repurposing secondary screen (2026): AUC normalised to 1
    - GDSC datasets:
        - Sanger GDSC1: AUC normalised to 1
        - Sanger GDSC2: AUC normalised to 1

In [45]:
print("GDSC cell lines included in CCLE:", len(gdsc_counts.isin(ccle_counts)))

GDSC cell lines included in CCLE: 54


-Drug names were normalised form comparasons

In [46]:
def normalize_drug_name(value):
    value = str(value).strip().upper()

    value = re.sub(
        r"\s+\((?:GDSC1|GDSC2|CTRP|BRD):[^)]*\)\s*$",
        "",
        value,
    )

    value = re.sub(r"\s+", " ", value)

    return value

In [47]:
drug_sets = {
    "gdsc1": set(gdsc1.columns.map(normalize_drug_name)),
    "gdsc2": set(gdsc2.columns.map(normalize_drug_name)),
    "ctd2": set(ctd2.columns.map(normalize_drug_name)),
    "prism": set(prism.columns.map(normalize_drug_name)),
    "ctrpv2": set(ctrpv2.columns.map(normalize_drug_name)),
}

for name, drugs in drug_sets.items():
    print(f"{name}: {len(drugs)} unique normalized drugs")

gdsc1: 316 unique normalized drugs
gdsc2: 286 unique normalized drugs
ctd2: 545 unique normalized drugs
prism: 1482 unique normalized drugs
ctrpv2: 481 unique normalized drugs


In [48]:
dataset_names = list(drug_sets)

overlap_counts = pd.DataFrame(
    index=dataset_names,
    columns=dataset_names,
    dtype=int,
)

for name1 in dataset_names:
    for name2 in dataset_names:
        overlap_counts.loc[name1, name2] = len(
            drug_sets[name1] & drug_sets[name2]
        )

overlap_counts

,gdsc1,gdsc2,ctd2,prism,ctrpv2
gdsc1,316.0,65.0,74.0,115.0,44.0
gdsc2,65.0,286.0,83.0,122.0,55.0
ctd2,74.0,83.0,545.0,167.0,436.0
prism,115.0,122.0,167.0,1482.0,132.0
ctrpv2,44.0,55.0,436.0,132.0,481.0


In [49]:
shared_by_all = set.intersection(
    *drug_sets.values()
)

print("Drugs shared by all five:", len(shared_by_all))
print(sorted(shared_by_all))

Drugs shared by all five: 23
['AFATINIB', 'ALISERTIB', 'AZD4547', 'AZD7762', 'BI-2536', 'BMS-345541', 'BMS-754807', 'BORTEZOMIB', 'CRIZOTINIB', 'DASATINIB', 'ENTINOSTAT', 'ERLOTINIB', 'FLUOROURACIL', 'FULVESTRANT', 'GEMCITABINE', 'LAPATINIB', 'MG-132', 'OLAPARIB', 'PACLITAXEL', 'RUXOLITINIB', 'SELUMETINIB', 'SIROLIMUS', 'SORAFENIB']


In [50]:
all_unique_drugs = set.union(
    *drug_sets.values()
)

raw_drug_count = sum(
    len(dataframe.columns)
    for dataframe in [
        gdsc1,
        gdsc2,
        ctd2,
        prism,
        ctrpv2,
    ]
)

print("Raw drug-column count:", raw_drug_count)
print("Unique normalized drugs:", len(all_unique_drugs))
print(
    "Repeated occurrences across datasets:",
    raw_drug_count - len(all_unique_drugs),
)

Raw drug-column count: 3110
Unique normalized drugs: 2228
Repeated occurrences across datasets: 882


In [51]:
print("CTD^2: (cell lines, drugs)", ctd2.shape)
print("CTRPv2: (cell lines, drugs)", ctrpv2.shape)
print("PRISM: (cell lines, drugs)", prism.shape)
print("GDSC1: (cell lines, drugs)", gdsc1.shape)
print("GDSC2: (cell lines, drugs)", gdsc2.shape)

CTD^2: (cell lines, drugs) (47, 545)
CTRPv2: (cell lines, drugs) (47, 481)
PRISM: (cell lines, drugs) (55, 1482)
GDSC1: (cell lines, drugs) (54, 316)
GDSC2: (cell lines, drugs) (54, 286)
